## 종합과제

### 1. 데이터 설명
지난 달(26년 8월)에 진행한 5km 및 10km의 기록을 1km의 각 구간당 페이스(1km당 걸린시간)로 정리하였습니다.
각 데이터는 시간 표현의 맞춰 6진수로 정리하였으며 5.30은 5분30초 페이스 입니다. 5km런닝의 6~10km 구간은 0.0으로 패딩처리하였습니다.


데이터는 직접 런닝한 기록이며, 런닝 어플에 표시된 기록을 캡처 및 정리해둔 사진을 바탕으로 ai에게 구간별 기록을 6진수로 정리해달라고 하였습니다.

사용 프롬포트 : 내 런닝 기록의 구간별 페이스를 6진수로 정리해주고 (예시: 5분30초 -> 5.30) 5키로 런닝의 경우 남은 구간은 0.0으로 패딩처리해줘.

### 2. Tensor shape 설계
데이터는 다음과 같이 2차원 Tensor 구조로 설계하였습니다.

[Session, Kilometer] 로 설계하였습니다.

따라서[15,10]으로 나타낼 수 있습니다.

### 3. 왜 그 Shape 인가?
첫 번째 차원으로는 총 15개의 런닝 세션(런닝 횟수)를 나타내고, 행 방향으로 구분합니다.

두 번째 차원으로는 15개의 러닝 세션 각각의 10키로미터의 1키로미터당 패이스 10개의 기록을 나타내기 때문에 10개의 열로 구분합니다.

더 많은 런닝 횟수와 거리가 있다면, shape의 크기는 유동적으로 증감 할 수 있습니다.

### 4. PyTorch Tensor 구현

In [2]:
# 4. PyTorch Tensor 구현
import torch

running_tensor = torch.tensor([
    [5.56, 5.46, 5.50, 5.40, 5.44, 5.52, 6.02, 6.04, 6.08, 6.10],  # 10km
    [6.08, 5.58, 5.52, 5.50, 5.46, 0.0, 0.0, 0.0, 0.0, 0.0],       # 5km
    [5.58, 5.56, 5.46, 5.50, 5.52, 6.02, 5.58, 6.08, 6.10, 6.20],  # 10km
    [5.44, 5.40, 5.38, 5.34, 5.40, 5.50, 5.52, 5.58, 6.02, 6.08],  # 10km
    [5.50, 5.40, 5.44, 5.38, 5.34, 0.0, 0.0, 0.0, 0.0, 0.0],       # 5km
    [5.46, 5.50, 5.44, 5.40, 5.46, 5.56, 5.58, 6.02, 6.04, 6.14],  # 10km
    [6.14, 6.04, 5.58, 5.52, 5.56, 0.0, 0.0, 0.0, 0.0, 0.0],       # 5km
    [6.02, 5.52, 5.56, 5.46, 5.50, 5.52, 6.02, 6.04, 6.08, 6.10],  # 10km
    [5.52, 5.56, 5.46, 5.50, 5.44, 5.46, 5.56, 5.58, 6.02, 6.04],  # 10km
    [5.56, 5.46, 5.50, 5.40, 5.44, 0.0, 0.0, 0.0, 0.0, 0.0],       # 5km
    [6.04, 6.02, 5.52, 5.56, 5.46, 5.52, 6.02, 5.58, 6.08, 6.10],  # 10km
    [5.50, 5.40, 5.44, 5.34, 5.38, 5.40, 5.50, 5.52, 5.56, 5.58],  # 10km
    [5.40, 5.38, 5.34, 5.32, 5.28, 0.0, 0.0, 0.0, 0.0, 0.0],       # 5km
    [5.58, 6.02, 5.52, 5.50, 5.46, 5.56, 5.58, 6.04, 6.08, 6.10],  # 10km
    [5.44, 5.34, 5.38, 5.28, 5.32, 5.34, 5.44, 5.46, 5.50, 5.52]   # 10km
])

print("Tensor Shape:", running_tensor.shape)

Tensor Shape: torch.Size([15, 10])


### 5. 결과값 3개 이상 계산
전체 평균 페이스

세션별 평균 페이스

최고 세션 페이스(최소시간 = 최솟값)

6진수를 평균을 구하는 방법과 패딩된 값을 처리하는 방법에 대해서 ai에 질문하였습니다.

#### 사용 프롬포트

1. 시간표현인 6진수의 평균을 구하는 방법 알려줘 -> 분에 60을 곱해 초 단위로 변경, 계산 한 후 다시 분으로 전환

2. 패딩된 값은 어떻게 처리해야 할까? -> valid_mask = running_tensor > 0.0 를 활용해 0.0을 제외하고 진행

In [3]:
# 5. 결과값 3개 이상 계산

#패딩 제외하기
valid_mask = running_tensor > 0.0

#분 -> 초
def pace_to_sec(pace):
    minutes = torch.floor(pace)
    seconds = torch.round((pace-minutes) * 100)
    return minutes * 60 +seconds

#초 -> 분
def pace_to_min(sec):
    minutes = torch.floor(sec / 60)
    seconds = torch.round(sec % 60)
    return minutes + seconds / 100

running_sec = pace_to_sec(running_tensor)
print("6진수 시간 -> 10진수 초 단위로 변경\n", running_sec,"\n")

#전체 평균 페이스 구하기
total_pace_sec = running_sec[valid_mask].mean()
total_pace = pace_to_min(total_pace_sec)

#세션별 평균 페이스 구하기
session_pace_sec = running_sec.sum(dim=1) / valid_mask.sum(dim=1)
session_pace = pace_to_min(session_pace_sec)

# 최고 세션 페이스 구하기
min_pace_sec = session_pace_sec.min()
min_pace = pace_to_min(min_pace_sec)

print(f"1. 전체 평균 페이스: {total_pace.item():.2f} \n")
print("2. 세션별 평균 페이스: ")
for i in range(len(session_pace)):
    print(f"{i+1}번째 세션: {session_pace[i].item():.2f}")
print(f"\n3. 최고 세션 페이스(최소시간 = 최솟값): {min_pace:.2f}")

6진수 시간 -> 10진수 초 단위로 변경
 tensor([[356., 346., 350., 340., 344., 352., 362., 364., 368., 370.],
        [368., 358., 352., 350., 346.,   0.,   0.,   0.,   0.,   0.],
        [358., 356., 346., 350., 352., 362., 358., 368., 370., 380.],
        [344., 340., 338., 334., 340., 350., 352., 358., 362., 368.],
        [350., 340., 344., 338., 334.,   0.,   0.,   0.,   0.,   0.],
        [346., 350., 344., 340., 346., 356., 358., 362., 364., 374.],
        [374., 364., 358., 352., 356.,   0.,   0.,   0.,   0.,   0.],
        [362., 352., 356., 346., 350., 352., 362., 364., 368., 370.],
        [352., 356., 346., 350., 344., 346., 356., 358., 362., 364.],
        [356., 346., 350., 340., 344.,   0.,   0.,   0.,   0.,   0.],
        [364., 362., 352., 356., 346., 352., 362., 358., 368., 370.],
        [350., 340., 344., 334., 338., 340., 350., 352., 356., 358.],
        [340., 338., 334., 332., 328.,   0.,   0.,   0.,   0.,   0.],
        [358., 362., 352., 350., 346., 356., 358., 364., 368., 37

### 6. 결과를 한 문장으로 해석

In [4]:
print(f"러닝 페이스 데이터는 {running_tensor.shape} 형태의 Tensor로 변환되었으며, 이를 기반으로 분석한 결과 전체 페이스는 {total_pace.item():.2f}이고, 최고 세션 페이스는 {min_pace.item():.2f}이다.")

러닝 페이스 데이터는 torch.Size([15, 10]) 형태의 Tensor로 변환되었으며, 이를 기반으로 분석한 결과 전체 페이스는 5.52이고, 최고 세션 페이스는 5.34이다.
